#### Face Identity Classification
This example illustrates Model Inversion (MINV) attacks on a face identity classfier model. The classfier is trained on [Large-scale CelebA - Aligned&Cropped](https://mmlab.ie.cuhk.edu.hk/projects/CelebA.html) dataset. Please ensure the following structure in the data folder:

directory_structure:

```
data/
    ├── private/          # Private dataset
    │   ├── identity_1/
    │   │   ├── instance_1.jpg
    │   │   ├── instance_2.jpg
    │   │   └── ...
    │   ├── identity_2/
    │   │   ├── instance_1.jpg
    │   │   ├── instance_2.jpg
    │   │   └── ...
    │   └── ...
    └── public/           # Public dataset
        ├── identity_1/
        │   ├── instance_1.jpg
        │   ├── instance_2.jpg
        │   └── ...
        ├── identity_2/
        │   ├── instance_1.jpg
        │   ├── instance_2.jpg
        │   └── ...
        └── ...      
```


To prepare data for this example:

1. Download CelebA images "img_align_celeba.zip" from this [URL](https://drive.google.com/drive/folders/0B7EVK8r0v71pTUZsaXdaSnZBZzg?resourcekey=0-rJlzl934LzC-Xp28GeIBzQ)
2. Download CelebA identities "identity_CelebA.txt" from this [URL](https://drive.google.com/drive/folders/0B7EVK8r0v71pOC0wOVZlQnFfaGs?resourcekey=0-pEjrQoTrlbjZJO2UL8K_WQ)
3. Unzip "img_align_celeba.zip" and place /img_align_celeba folder in examples/minv/celebA/data/
4. Place "identity_CelebA.txt" in examples/minv/celebA/data/
5. In the cell below select n classes to use, and n_private classes as private, n - n_private will be public.
6. Run the cell below

In [ ]:
# Path to the dataset and project root
import sys
import os

data_folder = "./data"
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.insert(0,project_root)

In [ ]:
# Description: This script partitions the CelebA dataset into public and private folders based on the identity of the person in the image.

import pandas as pd
import shutil
from tqdm import tqdm

# read img_align_celeba/identity_CelebA.txt
df = pd.read_csv(f'{data_folder}/identity_CelebA.txt', sep=' ', header=None)
df.columns = ['img', 'label']

# Parameters
n = 10177 # total number of classes to take, max 10177
n_private = 1000 # number of classes to be private, max 10177 and <= n. The rest will be public. PLG-MI paper uses 1000 for private and rest for public.

# Create folders
os.makedirs(f'{data_folder}', exist_ok=True)
os.makedirs(f'{data_folder}/private', exist_ok=True)
os.makedirs(f'{data_folder}/public', exist_ok=True)


if not os.path.exists(f"{data_folder}/img_align_celeba"):
    print(f"Please download the img_align_celeba.zip and extract it to {data_folder}/img_align_celeba")

# Copy images to folders based on label
for i in tqdm(range(1,n+1)):
    if i <= n_private:
        # folder name should be 0-indexed
        # so we subtract 1 from the label
        os.makedirs(f'{data_folder}/private/' + str(i-1), exist_ok=True)
        for img in df[df['label'] == i]['img']:          
            shutil.copy(f'{data_folder}/img_align_celeba/' + img, f'{data_folder}/private/' + str(i-1) + '/' + img)
    else:
        os.makedirs(f'{data_folder}/public/' + str(i-1), exist_ok=True)
        for img in df[df['label'] == i]['img']:
            shutil.copy(f'{data_folder}/img_align_celeba/' + img, f'{data_folder}/public/' + str(i-1) + '/' + img)

# Train the target model

In [ ]:
import yaml

from utils.celebA_data import get_celebA_train_test_loader, get_celebA_publicloader

# Load the config.yaml file
with open('train_config.yaml', 'r') as file:
    train_config = yaml.safe_load(file)

# Generate the dataset and dataloaders
path = os.path.join(os.getcwd(), train_config["data"]["data_dir"])

train_loader, test_loader = get_celebA_train_test_loader(train_config)
num_classes = train_loader.dataset.dataset.get_classes()

# Create the public dataloader
_ = get_celebA_publicloader(train_config)

Run this cell if you want to train the model, otherwise skip this and load it in cell below public loader

In [ ]:
import torch

from models import VGG16
from utils.resnet_model import create_trained_model_and_metadata

# Get number of classes from the train_loader
num_classes = train_loader.dataset.dataset.get_classes()
print(num_classes)

# Create the model
model = VGG16(num_classes=num_classes)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)

# Load the model
model.return_feature = False
train_acc, train_loss, test_acc, test_loss = create_trained_model_and_metadata(model,train_loader,test_loader, train_config)

In [ ]:
import matplotlib.pyplot as plt

# Plot training and test accuracy
plt.figure(figsize=(5, 4))

plt.subplot(1, 2, 1)
plt.plot(train_acc, label='Train Accuracy')
plt.plot(test_acc, label='Test Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy over Epochs')
plt.legend()

# Plot training and test loss
plt.subplot(1, 2, 2)
plt.plot(train_loss, label='Train Loss')
plt.plot(test_loss, label='Test Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Path to the dataset and project root
import sys
import os

data_folder = "./data"
project_root = os.path.abspath(os.path.join(os.getcwd(), "../../.."))
sys.path.insert(0,project_root)

from leakpro import LeakPro
from celebA_plgmi_handler import CelebA_InputHandler
config_path = "audit.yaml"


# Initialize the LeakPro object
leakpro = LeakPro(CelebA_InputHandler, config_path)

# Run the audit
results = leakpro.run_audit()

## Using the Diffusion model to sample images. 
  
After the pretraining or finetuning is done, it is possible to use the diffusion model to sample images outside the actual attack. In order to do that run the following cell. 

In [ ]:
import os
import sys
import importlib
from pathlib import Path

import yaml
import matplotlib.pyplot as plt

project_root = Path("../../..")
base_dir = project_root / "examples" / "minv" / "celebA"

sys.path.insert(0, str(project_root))

import leakpro.attacks.utils.diffusion_handler as diffusion_handler_mod
importlib.reload(diffusion_handler_mod)

from leakpro.attacks.utils.diff_mi.setup import DiffMiConfig
from leakpro.attacks.utils.diffusion_handler import DiffMiHandler


def resolve_diffmi_config(config_path: Path) -> DiffMiConfig:
    with config_path.open("r") as f:
        full_config = yaml.safe_load(f)

    diffmi_cfg = next(
        entry for entry in full_config["audit"]["attack_list"]
        if entry["attack"] == "diffmi"
    )
    diffmi_cfg = {k: v for k, v in diffmi_cfg.items() if k != "attack"}

    output_dir = (config_path.parent / full_config["audit"]["output_dir"]).resolve()
    save_path = output_dir / "attack_objects" / "diffusion_models"
    diffmi_cfg["save_path"] = str(save_path)

    pretrain_stem = diffmi_cfg["pretrain"]["save_name"]
    finetune_stem = diffmi_cfg["finetune"]["save_name"]
    pretrain_matches = sorted(save_path.glob(f"{pretrain_stem}*.pt"))
    finetune_matches = sorted(save_path.glob(f"{finetune_stem}*.pt"))

    if finetune_matches:
        diffmi_cfg["do_fine_tune"] = True
        diffmi_cfg["finetune"]["save_name"] = finetune_matches[-1].stem
    elif pretrain_matches:
        diffmi_cfg["do_fine_tune"] = False
        diffmi_cfg["pretrain"]["save_name"] = pretrain_matches[-1].stem
    else:
        raise FileNotFoundError(f"No Diff-MI checkpoints found in {save_path}")

    return DiffMiConfig(**diffmi_cfg)

config_path = base_dir / "audit.yaml"
output_path = "sampled_image.png"

cfg = resolve_diffmi_config(config_path)

cfg.pretrain.diffusion_steps = 30
cfg.finetune.diffusion_steps = 30

diff_handler = DiffMiHandler(handler=None, configs=cfg)
samples, labels = diff_handler.sample_from_diffusion_model(batch_size=1, label=0)

image = samples[0].permute(1, 2, 0).cpu().numpy()
plt.imshow(image)
plt.axis("off")
plt.savefig(output_path, bbox_inches="tight", pad_inches=0)

print(f"Saved sample for label {int(labels[0])} to {output_path}")
